## Einops tutorial part 2, deep learning

Based on https://einops.rocks/2-einops-for-deep-learning/

In [1]:
from einops import rearrange, reduce
import numpy as np

In [2]:
x = np.random.RandomState(42).normal(size=[10, 32, 100, 200])

In [3]:
import torch

In [4]:
x = torch.from_numpy(x)
x.require_grad = True

In [5]:
type(x), x.shape

(torch.Tensor, torch.Size([10, 32, 100, 200]))

In [6]:
# converting bchw to bhwc format is a common operation in OpenCV
# einops operation support deep learning frameworks

y = rearrange(x, "b c h w -> b h w c")
y.shape

torch.Size([10, 100, 200, 32])

## Backpropagation

In [7]:
# You can backpropagate through einops operations

y0 = x
y1 = reduce(y0, "b c h w -> b c", "max")
y2 = rearrange(y1, "b c -> c b")
y3 = reduce(y2, "c b -> ", "sum")

y3.backward() # This example doesn't work
print(reduce(x.grad, "b c h w -> ", "sum"))

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

## Meet `einops.asnumpy`

Converts tensors to numpy (and pulls from GPU if necessary)

In [9]:
from einops import asnumpy

print(type(y3))
y3_numpy = asnumpy(y3)
print(type(y3_numpy))

<class 'torch.Tensor'>
<class 'numpy.ndarray'>


## Common building blocks for deep learning

In [12]:
# Flattening is a common operation, frequently appears at the boundary between convolutional and fully-connected layers.

y = rearrange(x, "b c h w -> b (c h w)")
print(y.shape)

torch.Size([10, 640000])


In [13]:
# space-to-depth

y = rearrange(x, "b c (h h1) (w w1) -> b (h1 w1 c) h w", h1=2, w1=2)
print(y.shape)

torch.Size([10, 128, 50, 100])


In [14]:
# depth-to-space (reverse of the previous)
y = rearrange(x, "b (h1 w1 c) h w -> b c (h h1) (w w1)", h1=2, w1=2)
print(y.shape)

torch.Size([10, 8, 200, 400])


## Reductions

In [15]:
# Simple global average pooling

y = reduce(x, "b c h w -> b c", "mean")
print(y.shape)

torch.Size([10, 32])


In [18]:
# Max pooling with 2x2 kernel

y = reduce(x, "b c (h h1) (w w1) -> b c h w", "max", h1=2, w1=2)
print(y.shape)

torch.Size([10, 32, 50, 100])


In [21]:
# You can skip names for reduced axes

y = reduce(x, "b c (h 2) (w 2) -> b c h w", reduction="max")
print(y.shape)

torch.Size([10, 32, 50, 100])


## 1d, 2d and 3d pooling are defined in a similar way

In [26]:
# for sequential, 1d models, you'll probably want pooling over time

y = reduce(x, "(t 2) b c d -> t b c d", reduction="max")
print(y.shape)

torch.Size([5, 32, 100, 200])


In [27]:
# for volumetric models, all 3 dimensions are pooled

y = reduce(x, "b (x 2) (y 2) (z 2) -> b x y z", reduction="max")
print(y.shape)

torch.Size([10, 16, 50, 100])


## Squeeze and unsqueeze (expand_dims)

In [29]:
# models typically work only with batches,
# so top predict a single image ...
image = rearrange(x[0, :3], "c h w -> h w c")
# ... create a dummy 1-element axis ...
y = rearrange(image, "h w c -> () c h w")
# ... imagine you predicted this with a convolutional network for classification,
# we'll just flatten axes ...
predictions = rearrange(y, "b c h w -> b (c h w)")
# ... finally, decompose (remove) dummy axis
predictions = rearrange(predictions, "() classes -> classes")
print(predictions.shape)

torch.Size([60000])


## keepdmis-like behaviour for reductions

- empty decompositions `()` provides dimensions of length 1, which are broadcastable
- alternatively, you can use `1` to introduce new axis, that's a synonym to `()`

In [33]:
# per-channel mean-normalization for each image

y = x - reduce(x, "b c h w -> b c 1 1", "mean")
print(y.shape)

torch.Size([10, 32, 100, 200])


In [34]:
# per-channel mean-normalization for whole batch:

y = x - reduce(y, "b c h w -> 1 c 1 1", "mean")
print(y.shape)

torch.Size([10, 32, 100, 200])


## Stacking

In [39]:
# list of tensors
list_of_tensors = list(x)

In [40]:
# New axis (one that enumerates tensors) appears first on the left side of the expression.
# Just as if you were indexing list - first you'd get the tensor by index

tensors = rearrange(list_of_tensors, "b c h w -> b h w c")
tensors.shape

torch.Size([10, 100, 200, 32])

In [41]:
# or may be stack along last dimension?
tensors = rearrange(list_of_tensors, "b c h w -> h w c b")
tensors.shape

torch.Size([100, 200, 32, 10])

## Concatenation

In [43]:
# concatenate over the first dimension
tensors = rearrange(list_of_tensors, "b c h w -> (b h) w c")
tensors.shape

torch.Size([1000, 200, 32])

In [45]:
# or concatenate along the last dimension
tensors = rearrange(list_of_tensors, "b c h w -> h w (b c)")
tensors.shape

torch.Size([100, 200, 320])

## Shuffling within a dimension

In [47]:
# Channel shuffle (as it is drawn in shufflenet paper: https://arxiv.org/abs/1707.01083)
y = rearrange(x, "b (g1 g2 c) h w -> b (g2 g1 c) h w", g1=4, g2=4)
y.shape

torch.Size([10, 32, 100, 200])

In [48]:
# simpler version of channel shuffle
y = rearrange(x, "b (g c) h w -> b (c g) h w", g=4)
y.shape

torch.Size([10, 32, 100, 200])

## Split a dimension

In [49]:
# Here's a super-convenient trick.
# Example: when a network predicts several bboxes for each position
# Assume we got 8 bboxes, 4 coordinates each.
# To get coordinates into 4 separate variables, you move corresponding dimension to front and unpack tuple.
bbox_x, bbox_y, bbox_w, bbox_h = rearrange(x, "b (coord bbox) h w -> coord b bbox h w", coord=4, bbox=8)
# You can now operator on individual variables
max_bbox_area = reduce(bbox_w * bbox_h, "b bbox h w -> b h w", "max")
print(bbox_x.shape)
print(max_bbox_area.shape)


torch.Size([10, 8, 100, 200])
torch.Size([10, 100, 200])


## Getting into the weeds of tensors packing

(TODO)

## Shape parsing

In [53]:
from einops import parse_shape

In [ ]:
def convolve_2d(x):
    # imagine we have a simple 2d convolution with padding,
    # so output has same shape as input.
    return x

In [55]:
# imagine we are working with 3d data
x_5d = rearrange(x, "b c x (y z) -> b c x y z", z=20)
# but we have only 2d convolutions.
# That's not a problem, since we can apply
y = rearrange(x_5d, "b c x y z -> (b z) c x y")
y = convolve_2d(y)
# parse_shape does not just specify additional information, but verifies that all dimensions match
y = rearrange(y, "(b z) c x y -> b c x y z", **parse_shape(x_5d, "b c x y z"))



In [56]:
parse_shape(x_5d, "b c x y z")

{'b': 10, 'c': 32, 'x': 100, 'y': 10, 'z': 20}

In [57]:
# we can skip some dimensions by writing underscore
parse_shape(x_5d, "batch c _ _ _")

{'batch': 10, 'c': 32}

## Striding anything

how to convert any operation into a strided operation?
(like convolution with strides, aka dilated/atrous convolution)

In [59]:
# each image is split into subgrids, each subgrid now is a separate "image"
y = rearrange(x, "b c (h hs) (w ws) -> (hs ws b) c h w", hs=2, ws=2)
y = convolve_2d(y)
# pack subgrids back to an image
y = rearrange(y, "(hs ws b) c h w -> b c (h hs) (w ws)", hs=2, ws=2)
assert y.shape == x.shape

## Layers

For frameworks that prefer operating with layers, layers are available.

You'll need to import a proper one depending on your backend:

In [61]:
from einops.layers.torch import Rearrange, Reduce

Einops layers are identical to operations, and have same parameters.
(for the exception of first argument, which should be passed during call)

```python
layer = Rearrange(pattern, **axes_lengths)
layer = Reduce(pattern, reduction, **axes_lengths)

# apply layer to tensor
x = layer(x)
```

In [63]:
# Usually it's more convenient to use layers, not operations, to build models

from torch.nn import Sequential, Conv2d, MaxPool2d, Linear, ReLU
from einops.layers.torch import Reduce

model = Sequential(
    Conv2d(3, 6, kernel_size=5),
    MaxPool2d(kernel_size=2),
    Conv2d(6, 16, kernel_size=5),
    # combined pooling and flattening in a single step
    Reduce("b c (h 2) (w 2) -> b (c h w)", "max"),
    Linear(16 * 5 * 5, 120),
    ReLU(),
    Linear(120, 10)
)